## 提取 CIMA meta 中的基础协变量表（BGE_ID / CIMA_ID / 年龄 / SEX）

这个脚本会：

- 读取 `data/raw/CIMA/meta/招募汇总467例-整合问卷&临床信息-20220109 -无姓名.xlsx`
- 默认优先使用 `脱敏version` sheet
- 只保留这 4 列：
  - `BGE_ID`
  - `CIMA_ID`
  - `年龄`
  - `SEX`
- 性别编码按以下规则转换：
  - `女 = 0`
  - `男 = 1`

In [1]:
#!/usr/bin/env python3
import pandas as pd
from pathlib import Path

# ===== 路径设置（相对项目根目录）=====
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and PROJECT_ROOT.name != "CIMA_multiomics_regulation":
    PROJECT_ROOT = PROJECT_ROOT.parent

if PROJECT_ROOT.name != "CIMA_multiomics_regulation":
    raise RuntimeError("未找到项目根目录 CIMA_multiomics_regulation，请在项目目录内运行该脚本。")

INPUT_FILE = PROJECT_ROOT / "data/raw/CIMA/meta/招募汇总467例-整合问卷&临床信息-20220109 -无姓名.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "data/processed/CIMA/meta_inventory"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / "basic_id_sex_age.csv"

print(f"INPUT_FILE = {INPUT_FILE}")
print(f"exists = {INPUT_FILE.exists()}")

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"找不到输入文件: {INPUT_FILE}")

# ===== 读取 Excel =====
xls = pd.ExcelFile(INPUT_FILE)
sheet_name = "脱敏version" if "脱敏version" in xls.sheet_names else xls.sheet_names[0]
print("sheet_names =", xls.sheet_names)
print("use_sheet =", sheet_name)

# 不设 header，先按原始矩阵读取
df_raw = pd.read_excel(INPUT_FILE, sheet_name=sheet_name, header=None)
print("df_raw shape =", df_raw.shape)

# 根据你这张表目前的结构：
# 第 3 行开始（0-based index = 3）是正式数据
# 第 7/8/9/10 列（0-based = 7,8,9,10）分别是 BGE编号, CIMA ID, 性别, 年龄
START_ROW = 3
COL_BGE = 7
COL_CIMA = 8
COL_SEX = 9
COL_AGE = 10

df = df_raw.iloc[START_ROW:, [COL_BGE, COL_CIMA, COL_AGE, COL_SEX]].copy()
df.columns = ["BGE_ID", "CIMA_ID", "年龄", "性别_raw"]

# ===== 基础清洗 =====
for col in ["BGE_ID", "CIMA_ID", "年龄", "性别_raw"]:
    df[col] = df[col].astype("string").str.strip()

# 去掉全空行
df = df[~(
    df["BGE_ID"].isna() &
    df["CIMA_ID"].isna() &
    df["年龄"].isna() &
    df["性别_raw"].isna()
)].copy()

# 年龄转数值
df["年龄"] = pd.to_numeric(df["年龄"], errors="coerce")

# 性别编码：女=0，男=1
sex_map = {
    "女": 0,
    "男": 1,
    "female": 0,
    "male": 1,
    "F": 0,
    "M": 1,
    "f": 0,
    "m": 1,
    "0": 0,
    "1": 1,
}
df["SEX"] = df["性别_raw"].map(sex_map)

# 常见脏值再兜底一次
df.loc[df["性别_raw"].str.contains("女", na=False), "SEX"] = 1
df.loc[df["性别_raw"].str.contains("男", na=False), "SEX"] = 2

out = df[["BGE_ID", "CIMA_ID", "年龄", "SEX"]].copy()

# 去掉 BGE_ID 和 CIMA_ID 都空的行
out = out[~(out["BGE_ID"].isna() & out["CIMA_ID"].isna())].copy()

# 保存
out.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print("\n输出文件：", OUTPUT_FILE)
print("out shape =", out.shape)
print(out.head(10))

print("\nSEX value counts:")
print(out["SEX"].value_counts(dropna=False))

print("\n缺失统计:")
print(out.isna().sum())


INPUT_FILE = /data/work/CIMA_multiomics_regulation/data/raw/CIMA/meta/招募汇总467例-整合问卷&临床信息-20220109 -无姓名.xlsx
exists = True
sheet_names = ['详细数据-467人', '脱敏version']
use_sheet = 脱敏version
df_raw shape = (470, 90)

输出文件： /data/work/CIMA_multiomics_regulation/data/processed/CIMA/meta_inventory/basic_id_sex_age.csv
out shape = (467, 4)
            BGE_ID CIMA_ID  年龄  SEX
3   E-B21106356138  CIMA01  23    1
4   E-B21133356716  CIMA02  59    2
5   E-B21138997257  CIMA03  68    2
6   E-B21155258684  CIMA04  35    1
7   E-B21296798284  CIMA05  29    1
8   E-B21325448328  CIMA06  70    1
9   E-B21329853502  CIMA07  51    1
10  E-B21350053724  CIMA08  44    2
11  E-B21386838548  CIMA09  48    2
12  E-B21409666675  CIMA10  77    2

SEX value counts:
SEX
1    261
2    206
Name: count, dtype: int64

缺失统计:
BGE_ID     0
CIMA_ID    0
年龄         0
SEX        0
dtype: int64
